# 🏥 TrialMatch AI — Medical NER Fine-Tuning

**Objective**: Fine-tune BioBERT using **QLoRA (PEFT)** for medical Named Entity Recognition.

**Entities**: AGE, SEX, CONDITION, STAGE, BIOMARKER, MUTATION, MEDICATION, LAB_VALUE, ECOG, PRIOR_THERAPY, RESPONSE, COMORBIDITY, ALLERGY, METASTATIC_SITE, DRUG_INTERACTION

**Task**: Token classification (BIO tagging) — each token gets B-ENTITY, I-ENTITY, or O

**Method**: QLoRA (4-bit quantization + LoRA) via PEFT

**Runtime**: T4 GPU required

In [ ]:
# ═══ Step 1: Install ═══
!pip install -q transformers datasets evaluate accelerate peft bitsandbytes scikit-learn seqeval

In [ ]:
# ═══ Step 2: Verify GPU ═══
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB)")

In [ ]:
# ═══ Step 3: Upload Training Data ═══
from google.colab import files
print("Upload train.jsonl (from fine_tuning/data/medical_ner/)")
uploaded = files.upload()

In [ ]:
# ═══ Step 4: Configuration ═══
ENTITY_LABELS = [
    "O",
    "B-AGE", "I-AGE",
    "B-SEX", "I-SEX",
    "B-CONDITION", "I-CONDITION",
    "B-STAGE", "I-STAGE",
    "B-BIOMARKER", "I-BIOMARKER",
    "B-MUTATION", "I-MUTATION",
    "B-MEDICATION", "I-MEDICATION",
    "B-LAB_VALUE", "I-LAB_VALUE",
    "B-ECOG", "I-ECOG",
    "B-PRIOR_THERAPY", "I-PRIOR_THERAPY",
    "B-RESPONSE", "I-RESPONSE",
    "B-COMORBIDITY", "I-COMORBIDITY",
    "B-ALLERGY", "I-ALLERGY",
    "B-METASTATIC_SITE", "I-METASTATIC_SITE",
    "B-DRUG_INTERACTION", "I-DRUG_INTERACTION",
]

label2id = {label: i for i, label in enumerate(ENTITY_LABELS)}
id2label = {i: label for i, label in enumerate(ENTITY_LABELS)}

CONFIG = {
    "base_model": "dmis-lab/biobert-v1.1",
    "max_seq_length": 256,
    "num_labels": len(ENTITY_LABELS),
    
    # QLoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["query", "value"],
    
    # Training
    "num_epochs": 20,
    "batch_size": 16,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    
    "output_dir": "./medical_ner_qlora",
    "merged_dir": "./medical_ner_merged",
}

print(f"✅ Config ready")
print(f"   Entity types: {len(ENTITY_LABELS)} labels (BIO scheme)")
print(f"   Unique entity types: {(len(ENTITY_LABELS)-1)//2}")
print(f"   Method: QLoRA (4-bit) with LoRA r={CONFIG['lora_r']}")

In [ ]:
# ═══ Step 5: Load and Convert to BIO Format ═══
import json
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])

# Load JSONL
raw_examples = []
with open("train.jsonl", "r") as f:
    for line in f:
        raw_examples.append(json.loads(line))

print(f"Loaded {len(raw_examples)} examples")

def convert_to_bio(text, entities, tokenizer):
    """
    Convert character-level entity spans to BIO-tagged token labels.
    Handles subword tokenization alignment.
    """
    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=CONFIG["max_seq_length"],
        return_offsets_mapping=True,
    )
    
    offsets = encoding["offset_mapping"]
    labels = [-100] * len(offsets)  # -100 = ignore in loss (special tokens)
    
    # Sort entities by start position
    sorted_entities = sorted(entities, key=lambda e: e["start"])
    
    for i, (start, end) in enumerate(offsets):
        if start == 0 and end == 0:
            # Special token ([CLS], [SEP], [PAD])
            labels[i] = -100
            continue
        
        # Default: O (outside any entity)
        labels[i] = label2id["O"]
        
        # Check if this token overlaps with any entity
        for ent in sorted_entities:
            ent_start = ent["start"]
            ent_end = ent["end"]
            ent_label = ent["label"]
            
            if start >= ent_start and end <= ent_end:
                # Token is inside this entity
                if start == ent_start or (i > 0 and offsets[i-1][1] <= ent_start):
                    # First token of entity → B-tag
                    bio_label = f"B-{ent_label}"
                else:
                    # Continuation → I-tag
                    bio_label = f"I-{ent_label}"
                
                if bio_label in label2id:
                    labels[i] = label2id[bio_label]
                break
    
    # Remove offset_mapping (not needed for training)
    encoding.pop("offset_mapping")
    encoding["labels"] = labels
    
    return encoding

# Convert all examples
processed = []
for ex in raw_examples:
    enc = convert_to_bio(ex["text"], ex["entities"], tokenizer)
    processed.append(enc)

print(f"✅ Converted {len(processed)} examples to BIO token classification format")

# Show a sample
sample = processed[0]
tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"][:20])
labs = [id2label.get(l, "IGN") if l != -100 else "[IGN]" for l in sample["labels"][:20]]
print(f"\nSample (first 20 tokens):")
for t, l in zip(tokens, labs):
    if t not in ["[PAD]"]:
        print(f"   {t:20s} → {l}")

In [ ]:
# ═══ Step 6: Create Dataset ═══
from datasets import Dataset
from sklearn.model_selection import train_test_split

if len(processed) > 4:
    train_data, val_data = train_test_split(processed, test_size=0.2, random_state=42)
else:
    train_data = processed
    val_data = processed[:1]  # Use first example as val if too few

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

train_dataset.set_format("torch")
val_dataset.set_format("torch")

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# ═══ Step 7: Load Model + QLoRA ═══
from transformers import AutoModelForTokenClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForTokenClassification.from_pretrained(
    CONFIG["base_model"],
    num_labels=CONFIG["num_labels"],
    id2label=id2label,
    label2id=label2id,
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)

# LoRA adapter
lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = 100 * trainable / total

print(f"✅ Model + QLoRA ready")
print(f"   Trainable: {trainable:,} ({pct:.2f}%)")
print(f"   Frozen: {total-trainable:,} ({100-pct:.2f}%)")

In [ ]:
# ═══ Step 8: Metrics ═══
import numpy as np
from seqeval.metrics import classification_report, f1_score as seq_f1

def compute_ner_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    # Convert to label strings (skip -100)
    true_labels = []
    pred_labels = []
    
    for pred_seq, label_seq in zip(preds, labels):
        true_seq = []
        pred_seq_str = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                true_seq.append(id2label.get(int(l), "O"))
                pred_seq_str.append(id2label.get(int(p), "O"))
        true_labels.append(true_seq)
        pred_labels.append(pred_seq_str)
    
    f1 = seq_f1(true_labels, pred_labels, average="weighted", zero_division=0)
    return {"f1": f1}

print("✅ NER metrics ready (seqeval F1)")

In [ ]:
# ═══ Step 9: Train ═══
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=5,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_ner_metrics,
)

print("🚀 Starting NER fine-tuning...")
train_result = trainer.train()
print(f"\n✅ Done! Loss: {train_result.training_loss:.4f}")

In [ ]:
# ═══ Step 10: Evaluate ═══
eval_results = trainer.evaluate()
print("\n" + "═" * 50)
print("📊 NER EVALUATION RESULTS")
print("═" * 50)
for k, v in eval_results.items():
    print(f"   {k}: {v:.4f}" if isinstance(v, float) else f"   {k}: {v}")
print("═" * 50)

In [ ]:
# ═══ Step 11: Merge LoRA + Save ═══
from peft import PeftModel
import os

print("Merging LoRA adapter...")

base_model = AutoModelForTokenClassification.from_pretrained(
    CONFIG["base_model"],
    num_labels=CONFIG["num_labels"],
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.float16,
)

checkpoints = [d for d in os.listdir(CONFIG["output_dir"]) if d.startswith("checkpoint-")]
best = os.path.join(CONFIG["output_dir"], sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]) if checkpoints else CONFIG["output_dir"]

merged = PeftModel.from_pretrained(base_model, best).merge_and_unload()
merged.save_pretrained(CONFIG["merged_dir"])
tokenizer.save_pretrained(CONFIG["merged_dir"])

# Save metrics
import json
metrics = {
    "model": CONFIG["base_model"],
    "method": "QLoRA (PEFT)",
    "task": "Token Classification (NER)",
    "entity_types": 15,
    "bio_labels": len(ENTITY_LABELS),
    "lora_rank": CONFIG["lora_r"],
    "lora_alpha": CONFIG["lora_alpha"],
    "trainable_params_pct": round(pct, 2),
    "train_loss": round(train_result.training_loss, 4),
    "eval_results": {k: round(v, 4) if isinstance(v, float) else v for k, v in eval_results.items()},
    "train_samples": len(train_dataset),
}
with open(f"{CONFIG['merged_dir']}/training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
with open(f"{CONFIG['merged_dir']}/lora_config.json", "w") as f:
    json.dump({"r": CONFIG["lora_r"], "alpha": CONFIG["lora_alpha"], "method": "QLoRA"}, f, indent=2)

print(f"✅ Merged model saved to {CONFIG['merged_dir']}")
print(json.dumps(metrics, indent=2))

In [ ]:
# ═══ Step 12: Test Inference ═══
from transformers import pipeline as hf_pipeline

ner = hf_pipeline(
    "ner",
    model=CONFIG["merged_dir"],
    tokenizer=CONFIG["merged_dir"],
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1,
)

test_texts = [
    "62-year-old male with stage IIIA NSCLC, EGFR positive, on Osimertinib",
    "48yo female HER2+ breast cancer, ECOG 0, prior Trastuzumab 6 cycles",
    "Allergic to Penicillin, comorbidities include type 2 diabetes and CKD",
]

print("\n🧪 NER INFERENCE TEST")
print("─" * 60)
for text in test_texts:
    results = ner(text)
    print(f"\nInput: {text}")
    for ent in results:
        print(f"   [{ent['entity_group']:20s}] {ent['word']:25s} (score: {ent['score']:.3f})")

In [ ]:
# ═══ Step 13: Download ═══
import shutil

shutil.make_archive("medical_ner_v1", "zip", ".", CONFIG["merged_dir"])

print("\n" + "═" * 60)
print("📦 NER MODEL READY")
print("═" * 60)
for f in sorted(os.listdir(CONFIG["merged_dir"])):
    size = os.path.getsize(os.path.join(CONFIG["merged_dir"], f)) / 1024
    print(f"   {f} ({size:.1f} KB)")

print(f"\n🎯 Unzip into: trialmatch-ai/fine_tuning/models/medical_ner/")

from google.colab import files
files.download("medical_ner_v1.zip")